# Estimation: exercises

In these exercises you will use a small sample of voice onset time (VOT) measurements to estimate the parameters of a population distribution, and then test how well those estimates predict new data from a different set of languages. Along the way, you will see why estimates are only as good as the sample they come from.

Most of the code in this notebook is already written for you. Wherever you see `___` (three underscores), replace it with the correct code, then run the cell. Each blank has a `# HINT:` comment on the line above it.

Run the cells in order from top to bottom, since later cells use results from earlier ones.

In [ ]:
# numpy: numerical tools (we will use it to make a range of x-values)
import numpy as np
# pandas: data frames
import pandas as pd
# seaborn: statistical plots
import seaborn as sns
# matplotlib: lower-level plotting (we will use it to draw a curve on top of a seaborn plot)
import matplotlib.pyplot as plt
# norm: the normal distribution from scipy
from scipy.stats import norm

## Exercise 1: VOT in unaspirated stops

Data: Lisker & Abramson (1964), via Cho & Ladefoged (1999), Table I.

![](https://raw.githubusercontent.com/ethanweed/ExPsyLing/master/images/Cho%20and%20Ladefoged%20Table%201.png)

Voice onset time (VOT) is the time in milliseconds between the release of a stop consonant and the start of vocal fold vibration. The table shows the average VOT of the unaspirated stops /p/, /t/ and /k/ in seven languages.

Imagine we could measure unaspirated stops in *every* language in the world. That would be our **population**. If we are honest with ourselves, we probably can't scrape together the resources to do that, so we treat these seven languages as a **sample** and use it to **estimate** the parameters (the mean $\mu$ and standard deviation $\sigma$) of the population distribution. 

### 1a. Enter the data

Fill in the VOT values for /t/ and /k/ from the table. Each list should be in the same order as the languages.

In [ ]:
# Each key becomes a column; each list holds one value per language (one row per language)
vot_wide = pd.DataFrame({
    'language': ['Dutch', 'Puerto Rican Spanish', 'Hungarian', 'Cantonese', 'Eastern Armenian', 'Korean', 'Tamil'],
    'p': [10, 4, 2, 9, 3, 18, 12],
    # HINT: the /t/ row of Table I, as numbers separated by commas
    't': [___],
    # HINT: the /k/ row of Table I, as numbers separated by commas
    'k': [___],
})
vot_wide

The sums of the three columns should add up to 58, 102, and 218. We can run a quick check to make sure you entered the data correctly:

In [ ]:
# assert raises an error if the condition is False, and does nothing if it is True
assert vot_wide[['p', 't', 'k']].sum().tolist() == [58, 102, 218], "Check your numbers"
print("Correct!")

### 1b. Reshape and plot

We can start inspecting the data by looking at a histogram.
Our data are currently in "wide" format, but to plot them in `seaborn` we will need to reshape them into "long" format.
`melt()` turns a wide table into a long one (see [Chapter 7](https://ethanweed.github.io/pythonbook/03.03-pragmatic_matters.html#reshaping-from-wide-to-long-format) of LSP for more details).

In [ ]:
# id_vars: the column to keep as-is
# var_name: name for the new column holding the old column names (p, t, k)
# value_name: name for the new column holding the values
vot = vot_wide.melt(id_vars='language', var_name='place', value_name='VOT')
vot.head()

Now each row is one observation: one stop in one language. Plot all 21 VOT values in a histogram.

In [ ]:
# binwidth=5: each bar covers 5 ms
# HINT: data is the long data frame we just made; x is the name of the column with the VOT values (in quotes)
sns.histplot(data=___, x=___, binwidth=5)

### 1c. Estimate the population parameters

Our best estimate of the population mean $\hat{\mu}$ is the sample mean. For the population standard deviation $\hat{\sigma}$, we use the sample data, but divide by $N-1$ instead of $N$ (Bessel's correction). Conveniently, pandas' `.std()` does this by default.

For now, ignore place of articulation and treat all 21 values as one sample.

In [ ]:
# Estimate the population mean and standard deviation from the sample
# HINT: select the 'VOT' column of vot and use .mean()
mu_hat = ___
# HINT: select the 'VOT' column of vot and use .std()
sigma_hat = ___
# :.2f rounds to two decimal places
print(f"Estimated population mean (mu_hat): {mu_hat:.2f}")
print(f"Estimated population standard deviation (sigma_hat): {sigma_hat:.2f}")

If the population is normally distributed, these two numbers are all we need to describe it. Plot a normal distribution with your estimated parameters on top of the data. `norm.pdf(x, mean, sd)` gives the height of the curve at each `x`.

In [ ]:
# 200 evenly spaced x-values from -20 to 60 ms, to draw a smooth curve
x = np.linspace(-20, 60, 200)
# stat='density' scales the histogram so it is comparable to the curve
sns.histplot(data=vot, x='VOT', binwidth=5, stat='density')
# Draw the normal curve with our estimated parameters
# HINT: the two estimates you just calculated: mean first, then standard deviation
plt.plot(x, norm.pdf(x, ___, ___))

**Q:** Are the data symmetric? Where does the normal curve put probability that real VOTs can't have?

The data are somewhat right-skewed: most values are short, with a long tail towards longer VOTs (e.g. Korean /k/ at 47 ms). The normal curve also extends below 0 ms, but /p/, /t/ and /k/ are voiceless stops, and voiceless stops cannot have negative VOT.So the normal distribution is not the best model here. Durations are often better described by a right-skewed distribution such as the ex-Gaussian, which combines a normal distribution with an exponential tail. The lognormal distribution is another option. We will stick with the normal for now, because its parameters are easy to estimate and interpret, but keep in mind that the choice of distribution is itself an assumption.

### 1d. One population or three?

So far we have assumed that all unaspirated stops come from the same population. But VOT is known to vary with place of articulation. Plot the mean VOT for each place. `pointplot` shows the mean of each group as a point, with a line showing one standard deviation on either side of it (`errorbar='sd'`).

In [ ]:
# y: one row per group; hue: give each group its own color; errorbar='sd': error bars show the standard deviation
# HINT: the name of the column that holds p, t and k (in quotes), for both blanks
sns.pointplot(data=vot, x='VOT', y=___, hue=___, errorbar='sd')

Now estimate the mean and standard deviation separately for each place of articulation.

In [ ]:
# groupby splits the data by place; agg computes mean and std for each group
# HINT: the same column you used in the plot above (in quotes)
estimates = vot.groupby(___)['VOT'].agg(['mean', 'std'])
estimates

**Q:** Which place of articulation stands out? Should we estimate one population distribution or three?

The velar stops (/k/) stand out: their VOTs are clearly longer than those of /p/ and /t/ in every language. This is a well-known pattern: the further back in the mouth the closure, the longer the VOT. Since the places seem to differ systematically, it makes more sense to estimate a separate distribution for each. From now on, we focus on the velars.

Save the estimated mean and standard deviation for /k/, so we can use them in Exercise 2.

In [ ]:
# .loc[row, column] picks out a single value from a data frame
# HINT: the row label for the velar stop (in quotes)
mu_k = estimates.loc[___, 'mean']
# HINT: same row label as the line above
sigma_k = estimates.loc[___, 'std']
print(f"Velar stops, Table I: mu_hat = {mu_k:.2f}, sigma_hat = {sigma_k:.2f}")

## Exercise 2: Do our estimates predict new data?

Data: Cho & Ladefoged (1999), Table IV.

![](https://raw.githubusercontent.com/ethanweed/ExPsyLing/master/images/Cho%20and%20Ladefoged%20Table%204.png)

Cho & Ladefoged measured stops in 18 other languages. The velar column is the only one with a value for every language, and the velars stand out here too: in almost every language, velars (or uvulars) have the longest VOT.

If our estimates from Exercise 1 are good estimates of the population of VOT for velar stops in the world's languages, they should describe these new languages reasonably well. Let's find out.

When a language contrasts plain and aspirated stops, the aspirated stops are on a second line. Table I only had unaspirated stops, so for a fair comparison we will use only the **plain** stops. (Wari' has two plain velar stops, at 50 and 58 ms; we keep only the first.)

### 2a. Reshape the data

This time the data are entered for you, in the same wide format as the table. Empty cells are `None`. The Retroflex column is left out, because it is empty.

In [ ]:
vot4_wide = pd.DataFrame({
    'language': ['Aleut (Eastern)', 'Aleut (Western)', 'Apache', 'Apache', 'Banawa', 'Bowiri', 'Chickasaw',
                 'Dahalo', 'Defaka', 'Gaelic', 'Gaelic', 'Hupa', 'Hupa', 'Jalapa Mazatec', 'Jalapa Mazatec',
                 'Khonoma Angami', 'Khonoma Angami', 'Montana Salish', 'Navajo', 'Navajo', 'Tlingit', 'Tlingit',
                 'Tsou', "Wari'", 'Yapese'],
    # 'aspirated' for the second line of a language with a plain/aspirated contrast, otherwise 'plain'
    'series':   ['plain', 'plain', 'plain', 'aspirated', 'plain', 'plain', 'plain',
                 'plain', 'plain', 'plain', 'aspirated', 'plain', 'aspirated', 'plain', 'aspirated',
                 'plain', 'aspirated', 'plain', 'plain', 'aspirated', 'plain', 'aspirated',
                 'plain', 'plain', 'plain'],
    'bilabial': [None, None, 13, None, None, 17, 13, 20, 18, 13, 64, 11, None, None, None,
                 10, 83, 22, 12, None, None, None, 11, 19, 20],
    'dental':   [None, None, None, None, 22, None, None, 15, None, 22, 65, 16, 82, None, None,
                 9, 55, None, None, None, None, 120, None, 26, 22],
    'alveolar': [59, 76, 15, 58, None, 18, 22, 42, 20, None, None, None, None, 11, 63,
                 None, None, 24, 6, 130, 18, None, 17, None, None],
    'velar':    [75, 95, 31, 80, 44, 39, 36, 27, 30, 28, 73, 44, 84, 23, 80,
                 20, 91, 48, 45, 154, 28, 128, 28, 50, 56],
    'uvular':   [78, 92, None, None, None, None, None, None, None, None, None, 27, None, None, None,
                 None, None, 55, None, None, 30, 128, None, None, None],
})
vot4_wide.head()

Reshape the data into long format, just like in Exercise 1. This time there are two columns to keep as they are.

In [ ]:
# id_vars can be a list of columns to keep as-is
# HINT: the two columns that are not places of articulation (in quotes)
vot4 = vot4_wide.melt(id_vars=[___, ___], var_name='place', value_name='VOT')
# dropna() removes the rows with empty cells
vot4 = vot4.dropna()
vot4.head()

Now keep only the plain velar stops. `&` means "and": both conditions must be true.

In [ ]:
# HINT: the place is 'velar'; which series did Table I measure?
velar = vot4[(vot4['place'] == ___) & (vot4['series'] == ___)]
assert len(velar) == 18, "There should be 18 plain velar stops"
print("Correct!")

### 2b. Compare the new data with our prediction

Plot a histogram of the new velar VOTs, and draw the normal curve with the parameters we estimated from Table I on top.

In [ ]:
x = np.linspace(-20, 120, 300)
sns.histplot(data=velar, x='VOT', binwidth=5, stat='density')
# HINT: the velar estimates you saved at the end of Exercise 1
plt.plot(x, norm.pdf(x, ___, ___), label='estimated from Table I')
plt.legend()

If our estimates are good, about 95% of new values should fall within two standard deviations of the mean. What proportion of the new languages actually do?

In [ ]:
lower = mu_k - 2 * sigma_k
# HINT: look at how lower is calculated on the line above
upper = ___
# between() is True for values inside the range; the mean of True/False values is the proportion of Trues
inside = velar['VOT'].between(lower, upper).mean()
print(f"Predicted range: {lower:.1f} to {upper:.1f} ms")
print(f"Proportion of new values inside the range: {inside:.2f}")

Let's look at the individual languages. Combine the velar stops from both tables, and plot each language as a point, with the predicted range shaded.

In [ ]:
# assign() adds a column; here it records which table each value comes from
table1_k = vot[vot['place'] == 'k'].assign(source='Table I')
table4_k = velar.assign(source='Table IV')
# concat() stacks the two data frames on top of each other; sort so the plot is ordered by VOT
both = pd.concat([table1_k, table4_k]).sort_values('VOT')

plt.figure(figsize=(6, 8))
# axvspan() shades the area between two x-values
plt.axvspan(lower, upper, color='grey', alpha=0.2, label='predicted range')
# HINT: the column made with assign() that says which table each value comes from (in quotes)
sns.scatterplot(data=both, x='VOT', y='language', hue=___, s=80)
plt.legend()

**Q:** Which languages fall outside the predicted range? Do the Table I languages look like typical members of the new sample?

### 2c. Estimates from the new data

Now estimate the mean and standard deviation from the new data, and compare them with the estimates from Table I.

In [ ]:
comparison = pd.DataFrame({
    'Table I': [mu_k, sigma_k, 7],
    # HINT: the mean and standard deviation of the 'VOT' column of velar
    'Table IV': [___, ___, len(velar)],
}, index=['mean', 'std', 'N'])
comparison.round(2)

**Q:** How well did our estimates from Table I predict the new data? Why might they be off?

Not very well. The new mean is about 10 ms higher, and the new data are more than twice as spread out as Table I suggested, so only about 70% of the new languages fall inside the predicted range instead of 95% (the Aleut velars, at 75 and 95 ms, are far outside it). There are several reasons: Table I has only 7 languages, so the estimates are uncertain, and neither set of languages is a random sample of the world's languages. Lisker & Abramson's languages are mostly large, well-studied languages, while Cho & Ladefoged's are mostly endangered languages from other families. Our estimates are only as good as the sample, and the population it represents.